# Day 23 Advanced Data Engineering
Run one cell at a time with the lab guide. Configure catalog and schema first. Expected failures are caught and verified. Cleanup is disabled by default.

## 1 Prepare your workspace

In [ ]:
import re
from datetime import datetime, timezone
from pyspark.sql import functions as F

catalog = "REPLACE_WITH_WORKING_CATALOG"
schema = "student_01"  # Change to YOUR assigned schema.
assert catalog != "REPLACE_WITH_WORKING_CATALOG", "Set catalog"
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog)
assert re.fullmatch(r"student_(0[1-9]|1[0-4])", schema)
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
ns = f"`{catalog}`.`{schema}`"
def tn(name):
    assert re.fullmatch(r"[a-z_]+", name)
    return f"{ns}.`day23_{run_id}_{name}`"
owned = []
display(spark.sql("SELECT current_user(), current_catalog(), "
                  "current_schema()"))
print("Keep this run identifier:", run_id)


Expected result: Your own user, approved catalog and assigned schema are displayed. Copy the run identifier into your evidence notes.

## 2 Understand the source orders

In [ ]:
from decimal import Decimal
rows = [
    (101, "Bengaluru", Decimal("100.00")),
    (102, "Pune", Decimal("200.00")),
    (103, "Bengaluru", Decimal("300.00")),
    (104, "Chennai", Decimal("-50.00")),
    (105, "Pune", None),
    (106, " ", Decimal("80.00"))
]
raw = spark.createDataFrame(
    rows, "order_id INT, city STRING, amount DECIMAL(12,2)"
).withColumn("source_batch", F.lit("shop_batch_001")) 
raw = raw.withColumn("ingested_at", F.current_timestamp())
bronze = tn("bronze")
raw.write.format("delta").mode("errorifexists").saveAsTable(bronze)
owned.append(bronze)
assert spark.table(bronze).count() == 6
display(spark.table(bronze).orderBy("order_id"))


Expected result: Six rows are stored. Amount is DECIMAL(12,2), preserving two decimal places for money. source_batch and ingested_at provide basic traceability.

## 3 Validate and quarantine bad records

In [ ]:
data = spark.table(bronze).withColumn("city", F.trim("city"))
reason = (
    F.when(F.col("order_id").isNull(), "missing_order_id")
     .when(F.col("city").isNull() | (F.col("city") == ""),
           "missing_city")
     .when(F.col("amount").isNull(), "missing_amount")
     .when(F.col("amount") < 0, "negative_amount")
)
checked = data.withColumn("reject_reason", reason)
valid = checked.filter("reject_reason IS NULL").drop("reject_reason")
rejected = checked.filter("reject_reason IS NOT NULL")
assert data.select("order_id").distinct().count() == 6
assert valid.count() == 3
assert rejected.count() == 3
assert valid.count() + rejected.count() == data.count()
silver, quarantine = tn("silver"), tn("quarantine")
valid.write.format("delta").mode("errorifexists").saveAsTable(silver)
owned.append(silver)
rejected.write.format("delta").mode("errorifexists").saveAsTable(quarantine)
owned.append(quarantine)
display(spark.table(quarantine).select("order_id", "reject_reason"))


Expected result: Silver has 101, 102 and 103. Quarantine has 104 negative_amount, 105 missing_amount, and 106 missing_city. Total rows accounted for equals six.

In [ ]:
gold = tn("gold")
summary = spark.table(silver).groupBy("city").agg(
    F.count("*").alias("order_count"),
    F.sum("amount").alias("sales")
)
summary.write.format("delta").mode("errorifexists").saveAsTable(gold)
owned.append(gold)
assert summary.agg(F.sum("sales")).first()[0] == Decimal("600.00")
display(spark.table(gold).orderBy("city"))


## 4 Make updates safe to repeat

In [ ]:
merge_target = tn("merge_orders")
spark.table(silver).select("order_id", "city", "amount").write \
    .format("delta").mode("errorifexists").saveAsTable(merge_target)
owned.append(merge_target)
updates = spark.createDataFrame([
    (102, "Pune", Decimal("250.00")),
    (107, "Chennai", Decimal("250.00"))
], "order_id INT, city STRING, amount DECIMAL(12,2)")
assert updates.count() == updates.select("order_id").distinct().count()
updates.createOrReplaceTempView("day23_updates")
merge_sql = f"""
MERGE INTO {merge_target} AS t
USING day23_updates AS s ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET
  t.city = s.city, t.amount = s.amount
WHEN NOT MATCHED THEN INSERT (order_id, city, amount)
VALUES (s.order_id, s.city, s.amount)
"""
for attempt in (1, 2):
    spark.sql(merge_sql)
    current = spark.table(merge_target)
    assert current.count() == 4
    assert current.select("order_id").distinct().count() == 4
    assert current.agg(F.sum("amount")).first()[0] == Decimal("900.00")
    print("Attempt", attempt, "has 4 orders and total 900")
display(spark.table(merge_target).orderBy("order_id"))


Expected result: Both attempts produce the same logical result: 101=100, 102=250, 103=300 and 107=250. The total is 900. Table history can still receive new commits on a repeated operation.

## 5 Trace a Spark execution plan

In [ ]:
N = 100_000
assert 1000 <= N <= 1_000_000
perf = tn("perf_orders")
large = spark.range(N).select(
    F.col("id").alias("order_id"),
    (F.col("id") % 100).alias("customer_id"),
    (F.col("id") % 4).cast("int").alias("city_id"),
    ((F.col("id") % 100) + 1).cast("decimal(12,2)").alias("amount")
)
large.write.format("delta").mode("errorifexists").saveAsTable(perf)
owned.append(perf)
orders = spark.table(perf)
query = orders.filter("amount >= 50").groupBy("city_id").agg(
    F.sum("amount").alias("sales")
)
query.explain("formatted")
display(query.orderBy("city_id"))


Expected result: Four city groups appear. Locate Scan, Filter, Aggregate and any Exchange nodes. Actual plan labels depend on the runtime and execution engine.

## 6 Compare join strategies fairly

In [ ]:
from time import perf_counter
from statistics import median
cities = spark.createDataFrame([
    (0, "Bengaluru"), (1, "Pune"),
    (2, "Chennai"), (3, "Delhi")
], "city_id INT, city STRING")
def join_query(use_hint):
    lookup = F.broadcast(cities) if use_hint else cities
    return (orders.filter("amount >= 50")
        .select("city_id", "amount")
        .join(lookup, "city_id")
        .groupBy("city").agg(F.sum("amount").alias("sales")))
a, b = join_query(False), join_query(True)
a.explain("formatted")
b.explain("formatted")
def run(q):
    start = perf_counter()
    rows = q.orderBy("city").collect()  # Only 4 result rows.
    return perf_counter() - start, rows
run(a); run(b)  # Warm up both variants.
times = {"automatic": [], "broadcast": []}
for i in range(3):
    sequence = [("automatic", a), ("broadcast", b)]
    if i % 2: sequence.reverse()
    results = {}
    for name, q in sequence:
        elapsed, result = run(q)
        times[name].append(elapsed)
        results[name] = result
    assert results["automatic"] == results["broadcast"]
print({k: round(median(v), 3) for k, v in times.items()})


Expected result: Results are identical. Record both medians in seconds. If both plans already use broadcast, similar timings are expected and are a valid finding.

## 7 Explore partitions and skew

In [ ]:
eight = orders.repartition(8)
two = eight.coalesce(2)
def partition_sizes(df):
    return (df.select(F.spark_partition_id().alias("partition_id"))
            .groupBy("partition_id").count()
            .orderBy("partition_id"))
display(partition_sizes(eight))
display(partition_sizes(two))
assert eight.count() == two.count() == N


Expected result: Inspect the populated partition IDs and row counts. Data contents are unchanged. Exact distribution and the optimized execution plan can vary. This uses DataFrame APIs rather than RDD APIs for Standard compute compatibility.

In [ ]:
skewed = orders.withColumn("hot_key",
    F.when(F.col("order_id") < int(N * 0.9), F.lit(0))
     .otherwise((F.col("order_id") % 9 + 1).cast("int")))
display(skewed.groupBy("hot_key").count().orderBy("hot_key"))
display(partition_sizes(skewed.repartition(8, "hot_key")))


Expected result: hot_key 0 has 90000 rows with the default N. It concentrates much of the work in the keyed repartition. The remaining keys can share partitions because hashing does not guarantee one key per partition.

## 8 Reuse data and control memory

In [ ]:
cached = orders.filter("amount >= 50").select("city_id", "amount")
cached.cache()
try:
    count = cached.count()  # Populate the cache.
    print("Cached qualifying rows:", count)
    display(cached.groupBy("city_id").agg(F.sum("amount")))
    display(cached.groupBy("city_id").agg(F.avg("amount")))
finally:
    cached.unpersist()


Expected result: At N=100000, there are 51000 qualifying rows. The two reports reuse the filtered data. The finally block releases the cache even if a later report fails.

## 9 Inspect Delta file layout

In [ ]:
def layout(table):
    return spark.sql(f"DESCRIBE DETAIL {table}").select(
        "format", "numFiles", "sizeInBytes", "partitionColumns")
display(layout(perf))
count_before = spark.table(perf).count()
spark.sql(f"OPTIMIZE {perf}")
display(layout(perf))
assert spark.table(perf).count() == count_before == N


Expected result: Row count remains N. numFiles may decrease or stay unchanged. The table may already be compact due to automatic write optimizations. Do not repeatedly append tiny batches just to manufacture a dramatic screenshot.

Optional ZORDER cell: run only after checking the table is not liquid clustered and the feature is available on this compute.

In [ ]:
RUN_ZORDER = False  # Trainer may enable after layout review.
if RUN_ZORDER:
    spark.sql(f"OPTIMIZE {perf} ZORDER BY (customer_id)")
    display(spark.table(perf).filter("customer_id = 42")
            .agg(F.count("*").alias("orders"),
                 F.sum("amount").alias("sales")))


## 10 Enforce data quality at the table

In [ ]:
constrained = tn("constrained_orders")
spark.sql(f"""CREATE TABLE {constrained} (
  order_id INT NOT NULL,
  amount DECIMAL(12,2) NOT NULL
) USING DELTA""")
owned.append(constrained)
spark.sql(f"ALTER TABLE {constrained} ADD CONSTRAINT "
          "positive_amount CHECK (amount >= 0)")
spark.sql(f"INSERT INTO {constrained} VALUES (1, 100.00)")


In [ ]:
before = spark.table(constrained).count()
failed = False
try:
    spark.sql(f"INSERT INTO {constrained} VALUES (2, -50.00)")
except Exception as exc:
    failed = True
    print(type(exc).__name__, str(exc)[:700])
assert failed, "Expected the negative amount to be rejected"
assert spark.table(constrained).count() == before == 1


Expected result: The negative write fails and the table still contains only the valid row. Confirm the error mentions the amount constraint; an unrelated permission error is not evidence of successful constraint validation.

## 11 Test schema enforcement

In [ ]:
incompatible = spark.createDataFrame(
    [(3, {"text": "not_money"})],
    "order_id INT, amount STRUCT<text:STRING>"
)
failed = False
try:
    incompatible.write.format("delta").mode("append") \
        .saveAsTable(constrained)
except Exception as exc:
    failed = True
    print(type(exc).__name__, str(exc)[:700])
assert failed, "Expected an incompatible schema rejection"
assert spark.table(constrained).count() == 1
spark.table(constrained).printSchema()


Expected result: The append is rejected; amount remains DECIMAL(12,2) and the row count remains one. Inspect the error for an incompatible type/schema cause. Do not treat unrelated failures as a pass.

## 12 Evolve a schema deliberately

In [ ]:
evolution = tn("schema_orders")
spark.sql(f"""CREATE TABLE {evolution} (
  order_id INT, amount DECIMAL(12,2)
) USING DELTA""")
owned.append(evolution)
spark.sql(f"INSERT INTO {evolution} VALUES (1, 100.00)")
spark.sql(f"ALTER TABLE {evolution} ADD COLUMNS "
          "(payment_method STRING)")
spark.sql(f"INSERT INTO {evolution} "
          "(order_id, amount, payment_method) VALUES (2, 200.00, 'UPI')")
assert spark.table(evolution).count() == 2
assert spark.table(evolution).filter(
    "order_id = 1 AND payment_method IS NULL").count() == 1
display(spark.table(evolution).orderBy("order_id"))


In [ ]:
new_row = spark.createDataFrame(
    [(3, Decimal("300.00"), "CARD", "mobile_app")],
    "order_id INT, amount DECIMAL(12,2), "
    "payment_method STRING, channel STRING"
)
new_row.write.format("delta").mode("append") \
    .option("mergeSchema", "true").saveAsTable(evolution)
assert "channel" in spark.table(evolution).columns
assert spark.table(evolution).count() == 3
display(spark.table(evolution).orderBy("order_id"))


Expected result: channel is added for this approved write. Orders 1 and 2 have NULL channel; order 3 has mobile_app. Rerunning the append creates duplicates, so run it once per attempt.

## 13 Capture actual table versions

In [ ]:
history_table = tn("history_orders")
spark.sql(f"""CREATE TABLE {history_table} (
  order_id INT, amount DECIMAL(12,2)
) USING DELTA""")
owned.append(history_table)
spark.sql(f"INSERT INTO {history_table} VALUES (101, 100), (102, 200)")
def latest_version(table):
    return int(spark.sql(f"DESCRIBE HISTORY {table}")
               .orderBy(F.desc("version")).first()["version"])
v_before = latest_version(history_table)
print("Initial data version:", v_before)


In [ ]:
spark.sql(f"UPDATE {history_table} SET amount = 250 "
          "WHERE order_id = 102")
v_after = latest_version(history_table)
assert v_after > v_before
display(spark.sql(f"DESCRIBE HISTORY {history_table}")
        .select("version", "timestamp", "operation"))
display(spark.table(history_table).orderBy("order_id"))


Expected result: Current values are 101=100 and 102=250. History includes the initial data write and the update. Record both actual versions in your worksheet.

## 14 Read history and restore

In [ ]:
old = spark.sql(f"SELECT * FROM {history_table} "
                f"VERSION AS OF {v_before}")
current = spark.table(history_table)
assert old.filter("order_id = 102").first()["amount"] == Decimal("200")
assert current.filter("order_id = 102").first()["amount"] == Decimal("250")
display(old.orderBy("order_id"))
display(current.orderBy("order_id"))


In [ ]:
spark.sql(f"RESTORE TABLE {history_table} "
          f"TO VERSION AS OF {v_before}")
v_restore = latest_version(history_table)
assert v_restore > v_after
restored = spark.table(history_table)
assert restored.count() == 2
assert restored.filter("order_id = 102").first()["amount"] == Decimal("200")
assert restored.agg(F.sum("amount")).first()[0] == Decimal("300")
display(spark.sql(f"DESCRIBE HISTORY {history_table}")
        .select("version", "timestamp", "operation"))


Expected result: The current total is 300 and history includes a new RESTORE commit. The restore did not erase all intervening history.

## 15 Design the production pipeline

## 16 Record evidence and solve common problems

## 17 Review and clean up your own objects

In [ ]:
print("Tables created during this session:")
for table in owned:
    print(table)
CLEANUP = False  # Change only after saving your evidence.
if CLEANUP:
    expected_prefix = f"{ns}.`day23_{run_id}_"
    assert all(t.startswith(expected_prefix) for t in owned)
    for table in reversed(owned):
        spark.sql(f"DROP TABLE IF EXISTS {table}")
    print("Dropped only the listed lab tables")
